# MedMamba-XAI — Contrastive Domain Fine-Tuning (Kaggle)

This notebook fine-tunes the **domain_projector + backbone** of a pre-trained MedMamba checkpoint
to improve prototype routing accuracy (pathmnist was only 48.9%).

## Before running — Kaggle setup checklist

1. **Run cell 3 locally in VS Code** to upload `checkpoint_best.pt` and optionally the full MedMNIST dataset to Kaggle
   - Set `UPLOAD_DATA = True` (default) to upload the pre-built JPEG folders (~9.7 GB, saves ~15 min on Kaggle)
   - Set `UPLOAD_DATA = False` to upload only the checkpoint; cell 8 will download MedMNIST on Kaggle instead
2. **Enable GPU on Kaggle:** Settings → Accelerator → GPU T4 x2 (or P100)
3. **Internet ON on Kaggle:** Settings → Internet → On (needed for git clone + pip install)
4. **Add checkpoint dataset:** *+ Add Data* → search `medmamba-checkpoint` → Add
5. **Add data dataset (if uploaded):** *+ Add Data* → search `medmamba-data` → Add

## 1. [LOCAL ONLY] Upload checkpoint to Kaggle

**Run this cell locally in VS Code — skip it when running on Kaggle.**

Reads your `KGAT_` token from `MedMamba-XAI/.kaggle/access_token` (already there).
Creates a private Kaggle dataset named `medmamba-checkpoint` and uploads `checkpoint_best.pt` to it.
If the dataset already exists it pushes a new version.

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  RUN THIS CELL LOCALLY IN VS CODE — DO NOT RUN ON KAGGLE               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, json, shutil, zipfile
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────
UPLOAD_DATA = True   # set False to skip the ~9.7 GB dataset upload

# ── 1. Read KGAT_ token ───────────────────────────────────────────────────
TOKEN_FILE = Path("../.kaggle/access_token")
assert TOKEN_FILE.exists(), f"Token not found at: {TOKEN_FILE.resolve()}"
token = TOKEN_FILE.read_text().strip()
assert token.startswith("KGAT_"), f"Unexpected token format: {token[:12]}..."
print(f"Token loaded: {token[:16]}...")

# ── 2. Install/upgrade kaggle BEFORE setting env var or importing ─────────
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

# ── 3. Set KAGGLE_API_TOKEN, flush module cache, re-import ───────────────
# kaggle/__init__.py auto-calls authenticate() at import time and reads
# KAGGLE_API_TOKEN exactly once, so the env var must be set before import.
os.environ["KAGGLE_API_TOKEN"] = token
for k in [k for k in sys.modules if "kaggle" in k.lower()]:
    del sys.modules[k]

import kaggle
username = kaggle.api.get_config_value("username")
if not username:
    raise RuntimeError(
        "Authentication failed — check your KGAT_ token.\n"
        f"Verify: curl -X POST https://api.kaggle.com/v1/security.OAuthService/IntrospectToken "
        f"-H 'Content-Type: application/json' -d '{{\"token\":\"{token}\"}}'"
    )
print(f"Authenticated as: {username}")
api = kaggle.api

# ═══════════════════════════════════════════════════════════════════════════
# A. Upload checkpoint
# ═══════════════════════════════════════════════════════════════════════════
CKPT_PATH    = Path("../runs/medical_mamba/checkpoint_best.pt")
CKPT_SLUG    = "medmamba-checkpoint"
CKPT_STAGE   = Path("/tmp/kaggle_upload_ckpt")

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH.resolve()}"
CKPT_STAGE.mkdir(parents=True, exist_ok=True)

dst = CKPT_STAGE / "checkpoint_best.pt"
if dst.exists(): dst.unlink()
shutil.copy2(CKPT_PATH, dst)
print(f"\nCheckpoint staged: {dst.stat().st_size / 1e6:.0f} MB")

(CKPT_STAGE / "dataset-metadata.json").write_text(json.dumps({
    "title": "MedMamba Checkpoint",
    "id":    f"{username}/{CKPT_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}, indent=2))

existing = [d.ref for d in api.dataset_list(user=username)]
if f"{username}/{CKPT_SLUG}" in existing:
    print("Checkpoint dataset exists — uploading new version...")
    api.dataset_create_version(str(CKPT_STAGE), version_notes="checkpoint_best.pt",
                               quiet=False, convert_to_csv=False, delete_old_versions=False)
else:
    print("Creating checkpoint dataset...")
    api.dataset_create_new(str(CKPT_STAGE), public=False, quiet=False, convert_to_csv=False)

print(f"  → https://www.kaggle.com/datasets/{username}/{CKPT_SLUG}")
print(f"     mounts at /kaggle/input/{CKPT_SLUG}/checkpoint_best.pt")

# ═══════════════════════════════════════════════════════════════════════════
# B. Upload MedMNIST dataset (zipped JPEG folders, ~9.7 GB)
#    Kaggle has a 100k files/version limit, so each subset is zipped first.
#    Set UPLOAD_DATA = False at the top to skip this.
# ═══════════════════════════════════════════════════════════════════════════
if UPLOAD_DATA:
    DATA_SLUG  = "medmamba-data"
    DATA_STAGE = Path("/tmp/kaggle_upload_data")
    DATA_STAGE.mkdir(parents=True, exist_ok=True)
    LOCAL_DATA = Path("../dataset")
    SUBSETS    = ["pathmnist", "dermamnist", "bloodmnist", "octmnist"]

    print(f"\n── Zipping dataset folders (this may take 5-15 min) ─────────────")
    for s in SUBSETS:
        src = LOCAL_DATA / f"{s}_dataset"
        if not src.exists():
            print(f"  {s}: not found locally — skipping")
            continue
        zip_path = DATA_STAGE / f"{s}_dataset.zip"
        if zip_path.exists():
            print(f"  {s}: zip already exists ({zip_path.stat().st_size/1e6:.0f} MB) — reusing")
            continue
        n_files = sum(1 for _ in src.rglob("*.jpg"))
        print(f"  {s}: zipping {n_files:,} images...", flush=True)
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as z:
            for f in src.rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(src.parent))
        print(f"         → {zip_path.stat().st_size/1e6:.0f} MB")

    staged = list(DATA_STAGE.glob("*.zip"))
    if not staged:
        print("No dataset zips created — skipping data upload.")
    else:
        (DATA_STAGE / "dataset-metadata.json").write_text(json.dumps({
            "title": "MedMamba Dataset",
            "id":    f"{username}/{DATA_SLUG}",
            "licenses": [{"name": "CC0-1.0"}],
        }, indent=2))

        existing = [d.ref for d in api.dataset_list(user=username)]
        if f"{username}/{DATA_SLUG}" in existing:
            print(f"\nData dataset exists — uploading new version...")
            api.dataset_create_version(str(DATA_STAGE), version_notes="medmnist jpeg zips",
                                       quiet=False, convert_to_csv=False, delete_old_versions=False)
        else:
            print(f"\nCreating data dataset...")
            api.dataset_create_new(str(DATA_STAGE), public=False, quiet=False, convert_to_csv=False)

        print(f"  → https://www.kaggle.com/datasets/{username}/{DATA_SLUG}")
        print(f"     mounts at /kaggle/input/{DATA_SLUG}/")
else:
    print("\nUPLOAD_DATA=False — skipping dataset upload.")
    print("Cell 8 on Kaggle will download MedMNIST directly (~10-15 min).")

print("\n✓ Done. On Kaggle:")
print("  + Add Data → 'medmamba-checkpoint'  (checkpoint)")
if UPLOAD_DATA:
    print("  + Add Data → 'medmamba-data'        (pre-built JPEG dataset)")

Token loaded: KGAT_55207adf52e...



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Authenticated as: yusufafify

Checkpoint staged: 460 MB
Checkpoint dataset exists — uploading new version...
Starting upload for file checkpoint_best.pt


100%|██████████| 439M/439M [00:11<00:00, 38.7MB/s] 


Upload successful: checkpoint_best.pt (439MB)
  → https://www.kaggle.com/datasets/yusufafify/medmamba-checkpoint
     mounts at /kaggle/input/medmamba-checkpoint/checkpoint_best.pt

── Zipping dataset folders (this may take 5-15 min) ─────────────
  pathmnist: zipping 107,180 images...
         → 5474 MB
  dermamnist: zipping 10,015 images...
         → 336 MB
  bloodmnist: zipping 17,092 images...
         → 516 MB
  octmnist: zipping 109,309 images...
         → 3533 MB

Creating data dataset...
Starting upload for file octmnist_dataset.zip


...

Upload successful: bloodmnist_dataset.zip (493MB)
Starting upload for file dermamnist_dataset.zip


100%|██████████| 320M/320M [00:09<00:00, 37.2MB/s] 


Upload successful: dermamnist_dataset.zip (320MB)
Starting upload for file pathmnist_dataset.zip


100%|██████████| 5.10G/5.10G [02:07<00:00, 43.0MB/s]


Upload successful: pathmnist_dataset.zip (5GB)
  → https://www.kaggle.com/datasets/yusufafify/medmamba-data
     mounts at /kaggle/input/medmamba-data/

✓ Done. On Kaggle:
  + Add Data → 'medmamba-checkpoint'  (checkpoint)
  + Add Data → 'medmamba-data'        (pre-built JPEG dataset)


In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

# Clone the repo (contains src/medical_mamba package)
run("git clone --depth 1 https://github.com/yusufafify/MedMamba-XAI.git /kaggle/working/MedMamba-XAI")

# Install the medical_mamba package in editable mode + all deps
run(f"{sys.executable} -m pip install -q -e /kaggle/working/MedMamba-XAI")
run(f"{sys.executable} -m pip install -q medmnist einops")

print("Done.")

## 2. Paths & config

In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT    = Path("/kaggle/working/MedMamba-XAI")
DATA_ROOT    = Path("/kaggle/working/dataset")          # exploded JPEG datasets go here
CKPT_IN      = Path("/kaggle/input/medmamba-checkpoint/checkpoint_best.pt")
CKPT_OUT     = Path("/kaggle/working/checkpoint_contrastive.pt")  # final output
DATA_INPUT   = Path("/kaggle/input/medmamba-data")      # pre-uploaded zips (if available)

DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── Fine-tuning hyperparameters ────────────────────────────────────────────
EPOCHS             = 20
LR                 = 5e-5
CONTRASTIVE_LAMBDA = 0.5    # was 0.1 → backbone barely got domain gradient
CONTRASTIVE_TEMP   = 0.15   # was 0.07 → vanishing gradients in SupCon softmax
BATCH_SIZE         = 32
NUM_WORKERS        = 2
SEED               = 42

print(f"Checkpoint in : {CKPT_IN}")
print(f"Checkpoint out: {CKPT_OUT}")
print(f"Data root     : {DATA_ROOT}")
print(f"Data input    : {DATA_INPUT}  ({'found' if DATA_INPUT.exists() else 'not attached — will download'})")

assert CKPT_IN.exists(), (
    "Checkpoint not found! Upload checkpoint_best.pt as a Kaggle dataset "
    "and add it to this notebook. Expected path: " + str(CKPT_IN)
)

## 3. Download & explode MedMNIST datasets to JPEG folders

Downloads the four datasets via the `medmnist` package (resolution 224) and converts each `.npz`
archive to the flat `<split>/<index>_<label>.jpg` layout that `MedMNISTFolder` expects.
Skips any subset whose folder already exists.

In [ ]:
import os, zipfile, time
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import medmnist
from medmnist import INFO

SUBSETS = ["pathmnist", "dermamnist", "bloodmnist", "octmnist"]
RES = 224

# ── Fast path: use pre-uploaded zip files from Kaggle input ──────────────
if DATA_INPUT.exists():
    print(f"Pre-uploaded dataset found at {DATA_INPUT} — unzipping...")
    for s in SUBSETS:
        dst = DATA_ROOT / f"{s}_dataset"
        if (dst / "test").exists():
            print(f"  ✓ {s} already extracted — skipping")
            continue
        zip_src = DATA_INPUT / f"{s}_dataset.zip"
        if not zip_src.exists():
            print(f"  {s}: zip not in input — will download below")
            continue
        print(f"  Extracting {s} ({zip_src.stat().st_size/1e6:.0f} MB)...", flush=True)
        with zipfile.ZipFile(zip_src) as z:
            z.extractall(DATA_ROOT)
        print(f"    ✓ {sum(1 for _ in dst.rglob('*.jpg')):,} images")
    print("Unzip complete.\n")

# ── Slow path: download any subsets still missing ────────────────────────
def explode_npz(npz_path: Path, out_root: Path) -> None:
    """Convert a MedMNIST .npz archive to flat JPEG folders."""
    with zipfile.ZipFile(npz_path, "r") as arc:
        files = arc.namelist()
        for split in ["train", "val", "test"]:
            img_file = f"{split}_images.npy"
            lbl_file = f"{split}_labels.npy"
            if img_file not in files:
                continue
            split_dir = out_root / split
            split_dir.mkdir(parents=True, exist_ok=True)
            with arc.open(img_file) as f:
                version = np.lib.format.read_magic(f)
                shape, _, dtype = np.lib.format.read_array_header_1_0(f)
                with arc.open(lbl_file) as lf:
                    labels = np.load(lf)
                img_size = int(np.prod(shape[1:])) * dtype.itemsize
                print(f"  {split}: {shape[0]} images")
                for i in tqdm(range(shape[0]), desc=split, leave=False):
                    raw = f.read(img_size)
                    arr = np.frombuffer(raw, dtype=dtype).reshape(shape[1:])
                    dst_file = split_dir / f"{i}_{labels[i][0]}.jpg"
                    if dst_file.exists():
                        continue
                    Image.fromarray(arr).save(dst_file, "JPEG", quality=100)

NPZ_DIR = Path("/kaggle/working/medmnist_npz")
NPZ_DIR.mkdir(exist_ok=True)

for flag in SUBSETS:
    out_folder = DATA_ROOT / f"{flag}_dataset"
    if (out_folder / "test").exists():
        print(f"✓ {flag} already extracted — skipping")
        continue

    print(f"\n{'='*50}")
    print(f"Downloading & exploding {flag}...")

    npz_path = NPZ_DIR / f"{flag}_{RES}.npz"
    if not npz_path.exists():
        info = INFO[flag]
        DataClass = getattr(medmnist, info["python_class"])
        try:
            DataClass(split="train", download=True, size=RES, root=str(NPZ_DIR))
        except Exception as e:
            print(f"  (ignoring load error: {e})")
        candidates = list(NPZ_DIR.glob(f"{flag}*.npz"))
        if not candidates:
            raise FileNotFoundError(f"No .npz found for {flag} in {NPZ_DIR}")
        npz_path = candidates[0]

    explode_npz(npz_path, out_folder)
    print(f"  ✓ Done")

print("\nAll datasets ready.")
for flag in SUBSETS:
    d = DATA_ROOT / f"{flag}_dataset"
    n = sum(1 for _ in d.rglob("*.jpg"))
    print(f"  {flag:12s}: {n:,} images")

## 4. Load checkpoint & build model

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
import torch.nn as nn

from medical_mamba.data.constants import DATASET_META
from medical_mamba.models.medical_vmamba import build_model
from medical_mamba.utils.seed import set_seed

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

state = torch.load(CKPT_IN, map_location=device, weights_only=False)
task_names = state["task_names"]
cfg        = state.get("config", state.get("args", {}))

print(f"Tasks              : {task_names}")
print(f"Original epoch     : {state.get('epoch', '?')}")
print(f"Best avg F1        : {state.get('best_avg_f1', '?'):.4f}")
print(f"prototypes_computed: {state.get('prototypes_computed')}")

task_configs = [(n, DATASET_META[n]["num_classes"]) for n in task_names]
model = build_model(
    task_configs=task_configs,
    model_size=cfg.get("model_size", "tiny"),
    patch_size=cfg.get("patch_size", 8),
    head_dropout=cfg.get("head_dropout", 0.1),
)
missing, unexpected = model.load_state_dict(state["model_state_dict"], strict=False)
model.to(device)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Parameters: {n_params:.1f}M")
if missing:    print(f"Missing keys   : {len(missing)}")
if unexpected: print(f"Unexpected keys: {len(unexpected)}")

## 5. Freeze classification heads, unfreeze backbone + projector

In [ ]:
def freeze(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad_(False)

def unfreeze(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad_(True)

for name, module in model.named_children():
    if name in ("backbone", "domain_projector"):
        unfreeze(module)
        print(f"  TRAINABLE : {name}")
    else:
        freeze(module)
        print(f"  frozen    : {name}")

trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params) / 1e6
print(f"\nTrainable: {n_trainable:.1f}M / {n_params:.1f}M total")

## 6. Build dataloaders

In [ ]:
from medical_mamba.data.dataset import build_dataloaders
from medical_mamba.data.transforms import build_transforms_map

dataset_roots = {
    name: str(DATA_ROOT / f"{name}_dataset") for name in task_names
}
for name, path in dataset_roots.items():
    assert Path(path).exists(), f"Missing: {path}"

transforms_map = build_transforms_map(task_names)
loaders = build_dataloaders(
    dataset_roots=dataset_roots,
    transforms_map=transforms_map,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
print(f"Train: {len(loaders['train'].dataset):,} samples")
print(f"Val  : {len(loaders['val'].dataset):,} samples")

## 7. Fine-tuning loop

Pure SupCon loss — no CE. Classification heads are frozen so F1 scores are preserved.
The best backbone weights (lowest SupCon loss) are saved at the end.

In [ ]:
import time
import torch.optim as optim
import torch.nn.functional as F
from tqdm.auto import tqdm

from medical_mamba.training.losses import ContrastiveDomainLoss

criterion = ContrastiveDomainLoss(temperature=CONTRASTIVE_TEMP).to(device)

# Separate LRs: projector at 2× so SupCon gradient reaches backbone quicker
optimizer = optim.AdamW([
    {"params": list(model.backbone.parameters()),        "lr": LR,       "weight_decay": 0.05},
    {"params": list(model.domain_projector.parameters()),"lr": LR * 2.0, "weight_decay": 0.0},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR * 0.01
)

history = []
best_loss  = float("inf")
best_state = {}

print(f"Fine-tuning for {EPOCHS} epochs | lambda={CONTRASTIVE_LAMBDA} | temp={CONTRASTIVE_TEMP}")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    total, n, skipped = 0.0, 0, 0

    for batch in tqdm(loaders["train"], desc=f"Ep {epoch+1:02d}/{EPOCHS}", leave=False):
        images   = batch["image"].to(device, non_blocking=True)
        task_ids = batch["task_id"].to(device, non_blocking=True)

        # SupCon requires ≥2 distinct domains in the batch
        if task_ids.unique().numel() < 2:
            skipped += 1
            continue

        features, _  = model.backbone(images)
        projections  = model.project(features.float())
        loss = CONTRASTIVE_LAMBDA * criterion(projections, task_ids)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()

        total += loss.item()
        n += 1

    scheduler.step()
    avg = total / max(n, 1)
    lr_now = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0
    history.append({"epoch": epoch + 1, "loss": avg})

    marker = ""
    if avg < best_loss:
        best_loss = avg
        best_state = {
            k: v.clone()
            for k, v in model.state_dict().items()
            if "backbone" in k or "domain_projector" in k
        }
        marker = "  ← best"

    print(
        f"Ep {epoch+1:02d}/{EPOCHS} | loss={avg:.4f} | "
        f"lr={lr_now:.2e} | {elapsed:.0f}s | skipped={skipped}{marker}"
    )

print(f"\nBest SupCon loss: {best_loss:.4f}")

## 8. Plot training loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["epoch"], df["loss"], marker="o", lw=2, color="#1f77b4")
ax.axhline(best_loss, color="red", lw=1, linestyle="--", label=f"best={best_loss:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("SupCon loss × λ")
ax.set_title("Contrastive fine-tuning loss")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Recompute domain prototypes

Load best backbone weights back, then compute mean-feature prototypes from the **full training set**.
These replace the old prototypes that were computed from the weakly-separated feature space.

In [ ]:
print("Restoring best backbone weights...")
current_sd = model.state_dict()
current_sd.update(best_state)
model.load_state_dict(current_sd)

print("Computing domain prototypes from training set...")
model.eval()
model.compute_prototypes(loaders["train"], device)
print("Done.")

# Quick sanity check — cosine similarity between all prototype pairs.
# Well-separated prototypes should have low off-diagonal similarity (< 0.3).
protos = model.domain_prototypes  # (4, feat_dim)
import torch.nn.functional as F
protos_norm = F.normalize(protos, dim=1)
sim_matrix  = (protos_norm @ protos_norm.T).cpu().detach().numpy()

import seaborn as sns
import numpy as np

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    sim_matrix, annot=True, fmt=".3f", cmap="coolwarm",
    xticklabels=task_names, yticklabels=task_names,
    vmin=-1, vmax=1, ax=ax, square=True,
)
ax.set_title("Prototype cosine-similarity matrix\n(off-diagonal < 0.3 = good separation)")
plt.tight_layout()
plt.show()

off_diag = sim_matrix[~np.eye(len(task_names), dtype=bool)]
print(f"Off-diagonal similarity: mean={off_diag.mean():.3f}, max={off_diag.max():.3f}")
print("(lower = better domain separation, original run had poor separation for pathmnist)")

## 10. Validate routing accuracy on the test set

In [ ]:
import torchvision.transforms as T
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
from medical_mamba.data.dataset import MedMNISTFolder

# Build a test loader with averaged normalisation (same as predict.py)
means = [DATASET_META[n]["mean"] for n in DATASET_META]
stds  = [DATASET_META[n]["std"]  for n in DATASET_META]
avg_mean = [sum(m[i] for m in means) / len(means) for i in range(3)]
avg_std  = [sum(s[i] for s in stds)  / len(stds)  for i in range(3)]

avg_transform = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=avg_mean, std=avg_std),
])

test_sets = [
    MedMNISTFolder(
        dataset_name=name,
        root=str(DATA_ROOT / f"{name}_dataset"),
        split="test",
        transform=avg_transform,
        task_id=tid,
    )
    for tid, name in enumerate(task_names)
]
test_loader = DataLoader(
    ConcatDataset(test_sets), batch_size=64,
    num_workers=NUM_WORKERS, shuffle=False,
    pin_memory=(device.type == "cuda"),
)
print(f"Test samples: {sum(len(d) for d in test_sets):,}")

routing_true, routing_pred = [], []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Routing eval"):
        images    = batch["image"].to(device, non_blocking=True)
        true_tids = batch["task_id"].to(device, non_blocking=True)
        features, _ = model.backbone(images)
        sim = F.cosine_similarity(
            features.unsqueeze(1),
            model.domain_prototypes.unsqueeze(0),
            dim=-1,
        )
        pred_tids = sim.argmax(dim=-1)
        routing_true.extend(true_tids.cpu().numpy().tolist())
        routing_pred.extend(pred_tids.cpu().numpy().tolist())

routing_acc = sum(p == t for p, t in zip(routing_pred, routing_true)) / len(routing_true)
print(f"\nRouting accuracy: {routing_acc:.4f}  (baseline was 0.6545)")

# Confusion matrix
cm = confusion_matrix(routing_true, routing_pred, labels=list(range(len(task_names))))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_norm, annot=True, fmt=".3f", cmap="Purples",
    xticklabels=task_names, yticklabels=task_names,
    cbar_kws={"label": "Fraction of true-class samples"},
    ax=ax, square=True, vmin=0, vmax=1,
)
ax.set_xlabel("Predicted modality (prototype routing)")
ax.set_ylabel("True modality")
ax.set_title(f"Routing confusion matrix  |  accuracy = {routing_acc:.4f}")
plt.tight_layout()
plt.show()

# Per-modality recall
print("\nPer-modality routing recall:")
for i, name in enumerate(task_names):
    recall = cm_norm[i, i]
    base   = {"pathmnist": 0.489, "dermamnist": 0.922, "bloodmnist": 0.745, "octmnist": 0.998}.get(name, "?")
    arrow  = "↑" if recall > base else "↓" if recall < base else "="
    print(f"  {name:12s}: {recall:.3f}  (baseline {base})  {arrow}")

## 11. Save updated checkpoint

The saved file is a drop-in replacement for `checkpoint_best.pt`.
Download it from Kaggle output and place it at `runs/medical_mamba/checkpoint_contrastive.pt`.

In [ ]:
state["model_state_dict"]  = model.state_dict()
state["prototypes_computed"] = True
state["finetune_contrastive"] = {
    "epochs":    EPOCHS,
    "lambda":    CONTRASTIVE_LAMBDA,
    "temp":      CONTRASTIVE_TEMP,
    "best_loss": best_loss,
    "routing_acc_after": routing_acc,
}

torch.save(state, CKPT_OUT)
print(f"Saved → {CKPT_OUT}")
print(f"File size: {CKPT_OUT.stat().st_size / 1e6:.1f} MB")
print("\nDownload this file from Kaggle output and replace your local checkpoint.")